# 1-Model

`minimind-3` Dense 是 Decoder-Only Transformer，配置向 Qwen3 对齐，方便转 `transformers` / `llama.cpp` / `ollama` / `vllm`。完整实现在 `../model/model_minimind.py`，本本按模块拆开看，最后用一个极小配置做 forward / generate。

主线默认：`hidden_size=768`、`num_hidden_layers=8`、`q_heads=8`、`kv_heads=4`、`max_position_embeddings=32768`、`rope_theta=1e6`。MoE 版本是 `4 experts / top-1`，**没有 shared expert**。


In [ ]:
import os, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
print("cwd:", os.getcwd())

from model.model_minimind import (
    MiniMindConfig, RMSNorm, Attention, FeedForward, MOEFeedForward,
    MiniMindBlock, MiniMindModel, MiniMindForCausalLM,
    precompute_freqs_cis, apply_rotary_pos_emb, repeat_kv,
)
import torch, math
from torch import nn
print(MiniMindForCausalLM)
print(MiniMindConfig())


## MiniMindConfig

字段已对齐 Transformers / Qwen3 命名：`hidden_size`、`num_hidden_layers`、`num_attention_heads`、`num_key_value_heads`。MoE 相关项在 `use_moe=False` 时不会生效。


In [ ]:
cfg = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=False)
print("hidden_size", cfg.hidden_size)
print("layers", cfg.num_hidden_layers)
print("q/kv heads", cfg.num_attention_heads, cfg.num_key_value_heads)
print("head_dim", cfg.head_dim)
print("intermediate_size", cfg.intermediate_size)
print("rope_theta", cfg.rope_theta)
print("yarn", cfg.rope_scaling)
cfg_moe = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=True)
print("moe experts / topk", cfg_moe.num_experts, cfg_moe.num_experts_per_tok)


## RMSNorm

Pre-Norm：每个子层入口做 RMSNorm，没有减去均值，只按均方根缩放。


In [ ]:
x = torch.randn(2, 4, cfg.hidden_size)
print(RMSNorm(cfg.hidden_size, eps=cfg.rms_norm_eps)(x).shape)


## RoPE + YaRN

位置信息乘进 Q/K，而不是加到 embedding 上。主线用实数 cos/sin（不再用复数 polar）。推理时可开 YaRN：对超出训练长度的低频分量做插值，免训练外推长上下文。

`apply_rotary_pos_emb` 把向量后半部分取负再拼到前面，等价于二维旋转。


In [ ]:
cos, sin = precompute_freqs_cis(dim=cfg.head_dim, end=16, rope_base=cfg.rope_theta)
print(cos.shape, sin.shape)
q = torch.randn(2, 8, cfg.num_attention_heads, cfg.head_dim)
k = torch.randn(2, 8, cfg.num_key_value_heads, cfg.head_dim)
q2, k2 = apply_rotary_pos_emb(q, k, cos[:8], sin[:8])
print(q2.shape, k2.shape)


## Attention：GQA + QK-Norm

![GQA](./images/gqa.png)

- **GQA**：`8` 个 Q head 共享 `4` 个 KV head，`repeat_kv` 把 KV 扩到和 Q 一样多，省 KV cache。
- **QK-Norm**：投影之后、RoPE 之前，对每个 head 做 RMSNorm，小模型上更稳。
- 优先走 `scaled_dot_product_attention`（Flash Attention 路径）。


In [ ]:
attn = Attention(cfg)
pos = precompute_freqs_cis(cfg.head_dim, end=8, rope_base=cfg.rope_theta)
x = torch.randn(1, 8, cfg.hidden_size)
out, past = attn(x, pos, use_cache=True)
print("attn out", out.shape, "cached k", past[0].shape)


## SwiGLU FFN

`down(silu(gate(x)) * up(x))`。`intermediate_size` 默认按 `ceil(hidden_size * π / 64) * 64` 对齐。


In [ ]:
ffn = FeedForward(cfg)
print(ffn(torch.randn(1, 8, cfg.hidden_size)).shape)


## MiniMindBlock / MiniMindForCausalLM

一层 = Attention + FFN，都是 Pre-Norm 残差。`MiniMindForCausalLM` 继承 `GenerationMixin`，`forward(..., labels=)` 内部算 CE（`ignore_index=-100`），并返回 `aux_loss`。`tie_word_embeddings=True` 时 embedding 和 lm_head 共享权重。


In [ ]:
model = MiniMindForCausalLM(cfg)
input_ids = torch.randint(0, cfg.vocab_size, (1, 8))
labels = input_ids.clone()
out = model(input_ids, labels=labels)
print(type(out).__name__, "logits", out.logits.shape, "loss", float(out.loss), "aux", float(out.aux_loss))
print("params(M):", sum(p.numel() for p in model.parameters()) / 1e6)


## MoE：4 experts / top-1，无 shared expert

每个 token 经 router 选 top-k 个专家（默认 k=1）。训练时会算 load-balancing `aux_loss`。原生 PyTorch 实现下，专家一多 kernel 调度会很重，所以主线停在 4 专家这个甜点。


In [ ]:
moe_cfg = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=True)
moe = MiniMindForCausalLM(moe_cfg)
out = moe(input_ids, labels=labels)
print("dense params(M):", sum(p.numel() for p in model.parameters()) / 1e6)
print("moe params(M):", sum(p.numel() for p in moe.parameters()) / 1e6)
print("moe loss/aux", float(out.loss), float(out.aux_loss))


## 生成

`generate` 走 KV cache，支持 temperature / top-p / top-k。正式推理请用仓库根目录的 `eval_llm.py`。


In [ ]:
model.eval()
ids = model.generate(input_ids[:, :4], max_new_tokens=6, temperature=1.0)
print(ids)
